# Lab 10 — Data Preprocessing (Pandas)
This notebook mirrors `src/preprocessing.py` so you can view outputs step-by-step.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

repo_root = Path(__file__).resolve().parents[1] if '__file__' in globals() else Path('.').resolve()
csv_path = repo_root / 'data' / 'dataset.csv'
data = pd.read_csv(csv_path)
data.head()

In [ ]:
print('We have {} rows.'.format(data.shape[0]))
print('We have {} columns'.format(data.shape[1]))

In [ ]:
print('Total missing values:', int(np.sum(pd.isnull(data))))
print('Missing by column:')
print(data.isnull().sum()[data.isnull().sum() > 0])

In [ ]:
# Unique values example
if 'education' in data.columns:
    print(data['education'].unique())

In [ ]:
# Fill missing values using mode per column
for col in data.columns:
    if data[col].isnull().any():
        mode_val = data[col].mode(dropna=True).iloc[0]
        data[col] = data[col].fillna(mode_val)
print('Missing values after fill:', int(np.sum(pd.isnull(data))))

In [ ]:
# Drop id if present
if 'id' in data.columns:
    data.drop('id', axis=1, inplace=True)

# Convert float columns that look like integers into int
for col in data.columns:
    if data[col].dtype == 'float64':
        as_int = data[col].dropna().astype(np.int64)
        if np.allclose(as_int.astype(float), data[col].dropna().values):
            data[col] = data[col].astype(np.int64)

print(data.dtypes)

In [ ]:
TARGET_COL = 'loan_status'
y = data[TARGET_COL]
x = data.drop(columns=[TARGET_COL])
print('X shape:', x.shape)
print('y shape:', y.shape)

In [ ]:
# Convert object columns to integer codes
cat_columns = x.select_dtypes(['object']).columns
x[cat_columns] = x[cat_columns].apply(lambda s: pd.factorize(s)[0])

cleaned = x.copy()
cleaned[TARGET_COL] = y.values
cleaned.head()

In [ ]:
out_path = repo_root / 'outputs' / 'cleaned_dataset.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(out_path, index=False)
print('Saved:', out_path)